# ADNI no-MCI train latent longitudinal trends

This notebook analyzes only the `train` split for the no-MCI longitudinal experiment.

- It uses the checkpoint's learned training subject anchors directly, not anchor refits.
- It samples `20` training subjects with a random seed while cycling through `5`-year baseline age bins.
- It evaluates existing longitudinal future scans on the training subjects, summarizes forecast error, computes train-set velocity trends, shows where local change is strongest, and sweeps both in-distribution and out-of-distribution ages.


In [1]:
from IPython.display import display

import adni_no_mci_train_latent_trend_helpers as train_helpers

CHECKPOINT = '1000'  # change to 'selected' or another epoch if needed
DEVICE = 'auto'
N_SUBJECTS = 20
SEED = 7
BIN_WIDTH_YEARS = 5.0
MESH_RESOLUTION = 96
METRIC_SURFACE_SAMPLES = 12000
VELOCITY_METHOD = 'finite_difference'
SWEEP_END_AGE = 110.0
SWEEP_STEP_YEARS = 1.0
SWEEP_REFERENCE_AGE = 70.0

print('Experiment dir:', train_helpers.EXPERIMENT_DIR)
print('Checkpoint:', CHECKPOINT)
print('Subjects:', N_SUBJECTS, 'seed:', SEED)


Experiment dir: /home/jakaria/INR/Deep3DComp/examples/ADNI_1_L_No_MCI/longitudinal_age_disease_conditioned_cocycle_shape_multiple_pairs_real_pair_shape_reconstruction
Checkpoint: 1000
Subjects: 20 seed: 7


## Selected train subjects and observed volume trajectories

The selection is random but stratified by `5`-year baseline age bins through a round-robin pass over available bins.


In [2]:
overview = train_helpers.selected_train_subject_overview(
    n_subjects=N_SUBJECTS,
    seed=SEED,
    bin_width_years=BIN_WIDTH_YEARS,
)

display(
    overview['subject_df'][[
        'subject_id',
        'baseline_age_years',
        'age_bin_label',
        'num_scans',
        'first_diagnosis',
        'last_diagnosis',
        'age_span_years',
    ]].round(3)
)
display(overview['age_bin_counts_df'])

for fig in train_helpers.selected_train_overview_figures(
    n_subjects=N_SUBJECTS,
    seed=SEED,
    bin_width_years=BIN_WIDTH_YEARS,
):
    fig.show()


,subject_id,baseline_age_years,age_bin_label,num_scans,first_diagnosis,last_diagnosis,age_span_years
0,137_S_0366,57.0,55-60,3,AD,AD,1.0
1,036_S_0672,62.0,60-65,3,CN,CN,1.0
2,031_S_0321,68.0,65-70,3,AD,AD,1.0
3,057_S_0643,71.0,70-75,3,CN,CN,1.0
4,114_S_0601,77.0,75-80,3,CN,CN,1.0
5,128_S_0310,83.0,80-85,3,AD,AD,1.0
6,128_S_0266,86.0,85-90,3,AD,AD,1.0
7,002_S_0685,90.0,90-95,3,CN,CN,1.0
8,037_S_0627,59.0,55-60,3,AD,AD,1.0
9,067_S_0029,64.0,60-65,3,AD,AD,1.0


,age_bin_label,first_diagnosis,num_subjects
0,55-60,AD,3
1,60-65,AD,2
2,60-65,CN,1
3,65-70,AD,3
4,70-75,CN,3
5,75-80,AD,1
6,75-80,CN,2
7,80-85,AD,1
8,80-85,CN,1
9,85-90,AD,1


## Train-set existing future-shape forecast error

For each selected subject, this evaluates every existing future visit pair. The source latent is the checkpoint's learned training latent for the source scan.

- `direct`: one-shot latent transport from the source visit to the target visit
- `composed`: sequential transport through the observed intermediate visits
- `no_change`: keep the source shape unchanged as a baseline


In [3]:
forecast_payload = train_helpers.evaluate_selected_train_forecasts(
    checkpoint=CHECKPOINT,
    device=DEVICE,
    n_subjects=N_SUBJECTS,
    seed=SEED,
    bin_width_years=BIN_WIDTH_YEARS,
    mesh_resolution=MESH_RESOLUTION,
    metric_surface_samples=METRIC_SURFACE_SAMPLES,
)

display(
    forecast_payload['summary_df'][[
        'method',
        'pair_kind',
        'cohort',
        'num_rows',
        'num_subjects',
        'mesh_valid_fraction',
        'chamfer_aligned_mean',
        'assd_aligned_mean',
        'hd95_aligned_mean',
        'pred_minus_target_volume_mean',
        'latent_l2_to_target_state_mean',
    ]].round(6)
)

display(
    forecast_payload['forecast_df'][[
        'subject_id',
        'diagnosis',
        'source_scan_id',
        'target_scan_id',
        'pair_kind',
        'horizon_years',
        'method',
        'mesh_valid',
        'chamfer_aligned',
        'pred_minus_target_volume',
        'latent_l2_to_target_state',
    ]].head(30).round(6)
)


,method,pair_kind,cohort,num_rows,num_subjects,mesh_valid_fraction,chamfer_aligned_mean,assd_aligned_mean,hd95_aligned_mean,pred_minus_target_volume_mean,latent_l2_to_target_state_mean
0,direct,all,all,60,20,1.0,0.000153,0.007969,0.014264,-0.000073,0.005183
1,direct,all,CN,27,9,1.0,0.000159,0.008127,0.014637,-0.000097,0.005496
2,direct,all,AD,33,11,1.0,0.000147,0.007839,0.013959,-0.000054,0.004927
3,direct,adjacent,all,40,20,1.0,0.000156,0.008064,0.014455,-0.000056,0.007774
4,direct,adjacent,CN,18,9,1.0,0.000166,0.008272,0.014924,-0.000146,0.008245
5,direct,adjacent,AD,22,11,1.0,0.000149,0.007893,0.014071,0.000018,0.007390
6,direct,non_adjacent,all,20,20,1.0,0.000145,0.007780,0.013882,-0.000107,0.000000
7,direct,non_adjacent,CN,9,9,1.0,0.000147,0.007838,0.014062,0.000002,0.000000
8,direct,non_adjacent,AD,11,11,1.0,0.000143,0.007733,0.013734,-0.000196,0.000000
9,composed,all,all,60,20,1.0,0.000160,0.008143,0.014602,0.000011,0.010366


,subject_id,diagnosis,source_scan_id,target_scan_id,pair_kind,horizon_years,method,mesh_valid,chamfer_aligned,pred_minus_target_volume,latent_l2_to_target_state
0,002_S_0685,CN,ADNI_002_S_0685_MR_Hippocampal_Mask_Hi_2008022...,ADNI_002_S_0685_MR_Hippocampal_Mask_Hi_2008022...,adjacent,0.5,composed,True,0.000152,0.000460,0.000000
1,002_S_0685,CN,ADNI_002_S_0685_MR_Hippocampal_Mask_Hi_2008022...,ADNI_002_S_0685_MR_Hippocampal_Mask_Hi_2008022...,adjacent,0.5,direct,True,0.000152,0.000460,0.000000
2,002_S_0685,CN,ADNI_002_S_0685_MR_Hippocampal_Mask_Hi_2008022...,ADNI_002_S_0685_MR_Hippocampal_Mask_Hi_2008022...,adjacent,0.5,no_change,True,0.000422,0.009266,0.058195
3,002_S_0685,CN,ADNI_002_S_0685_MR_Hippocampal_Mask_Hi_2008022...,ADNI_002_S_0685_MR_Hippocampal_Mask_Hi_2008022...,non_adjacent,1.0,composed,True,0.000137,0.000568,0.008407
4,002_S_0685,CN,ADNI_002_S_0685_MR_Hippocampal_Mask_Hi_2008022...,ADNI_002_S_0685_MR_Hippocampal_Mask_Hi_2008022...,non_adjacent,1.0,direct,True,0.000136,0.000347,0.000000
5,002_S_0685,CN,ADNI_002_S_0685_MR_Hippocampal_Mask_Hi_2008022...,ADNI_002_S_0685_MR_Hippocampal_Mask_Hi_2008022...,non_adjacent,1.0,no_change,True,0.000387,0.010255,0.118050
6,002_S_0685,CN,ADNI_002_S_0685_MR_Hippocampal_Mask_Hi_2008022...,ADNI_002_S_0685_MR_Hippocampal_Mask_Hi_2008022...,adjacent,0.5,composed,True,0.000137,0.000568,0.008407
7,002_S_0685,CN,ADNI_002_S_0685_MR_Hippocampal_Mask_Hi_2008022...,ADNI_002_S_0685_MR_Hippocampal_Mask_Hi_2008022...,adjacent,0.5,direct,True,0.000137,0.000568,0.008407
8,002_S_0685,CN,ADNI_002_S_0685_MR_Hippocampal_Mask_Hi_2008022...,ADNI_002_S_0685_MR_Hippocampal_Mask_Hi_2008022...,adjacent,0.5,no_change,True,0.000312,0.000990,0.059858
9,016_S_0538,CN,ADNI_016_S_0538_MR_Hippocampal_Mask_Hi_2008022...,ADNI_016_S_0538_MR_Hippocampal_Mask_Hi_2008022...,adjacent,0.5,composed,True,0.000142,-0.000330,0.000000


In [4]:
for fig in train_helpers.train_forecast_summary_figures(
    checkpoint=CHECKPOINT,
    device=DEVICE,
    n_subjects=N_SUBJECTS,
    seed=SEED,
    bin_width_years=BIN_WIDTH_YEARS,
):
    fig.show()


## Representative train forecast examples

These overlays pick one `CN` and one `AD` training subject with a relatively long future horizon and show the source shape, the real future target shape, and the direct/composed prediction.


In [5]:
for fig in train_helpers.representative_train_forecast_figures(
    checkpoint=CHECKPOINT,
    device=DEVICE,
    n_subjects=N_SUBJECTS,
    seed=SEED,
    bin_width_years=BIN_WIDTH_YEARS,
    mesh_resolution=MESH_RESOLUTION,
):
    fig.show()


## Train latent velocity trends

This computes instantaneous normal velocity on the actual training scan meshes using the checkpoint's learned train subject anchors transported to each scan time.


In [6]:
velocity_payload = train_helpers.compute_selected_train_velocities(
    checkpoint=CHECKPOINT,
    device=DEVICE,
    n_subjects=N_SUBJECTS,
    seed=SEED,
    bin_width_years=BIN_WIDTH_YEARS,
    velocity_method=VELOCITY_METHOD,
)

display(velocity_payload['age_bin_df'].round(6))
display(
    velocity_payload['velocity_df'][[
        'subject_id',
        'scan_id',
        'diagnosis',
        'age_years',
        'volume',
        'area_weighted_rms_speed_per_year',
        'mean_absolute_speed_per_year',
        'net_volume_rate_per_year',
    ]].head(30).round(6)
)


,age_bin_center,diagnosis,num_rows,num_subjects,area_weighted_rms_speed_per_year_mean,mean_absolute_speed_per_year_mean,net_volume_rate_per_year_mean
0,57.5,all,8,3,0.020348,0.016108,0.000773
1,57.5,AD,8,3,0.020348,0.016108,0.000773
2,62.5,all,9,4,0.022925,0.017590,-0.003500
3,62.5,CN,3,1,0.027568,0.020277,-0.002399
4,62.5,AD,6,3,0.020604,0.016247,-0.004050
5,67.5,all,9,4,0.022035,0.017248,-0.004319
6,67.5,AD,9,4,0.022035,0.017248,-0.004319
7,72.5,all,9,4,0.020581,0.015878,-0.001832
8,72.5,CN,8,3,0.020561,0.015857,-0.000953
9,72.5,AD,1,1,0.020740,0.016048,-0.008864


,subject_id,scan_id,diagnosis,age_years,volume,area_weighted_rms_speed_per_year,mean_absolute_speed_per_year,net_volume_rate_per_year
0,002_S_0685,ADNI_002_S_0685_MR_Hippocampal_Mask_Hi_2008022...,CN,90.0,0.147767,0.015470,0.012067,0.002789
1,002_S_0685,ADNI_002_S_0685_MR_Hippocampal_Mask_Hi_2008022...,CN,90.5,0.138501,0.024198,0.018418,-0.016022
2,002_S_0685,ADNI_002_S_0685_MR_Hippocampal_Mask_Hi_2008022...,CN,91.0,0.137512,0.022141,0.017055,-0.009226
3,016_S_0538,ADNI_016_S_0538_MR_Hippocampal_Mask_Hi_2008022...,CN,83.0,0.161383,0.015680,0.012204,-0.002054
4,016_S_0538,ADNI_016_S_0538_MR_Hippocampal_Mask_Hi_2008022...,CN,83.5,0.150937,0.022303,0.017409,0.008873
5,016_S_0538,ADNI_016_S_0538_MR_Hippocampal_Mask_Hi_2008022...,CN,84.0,0.164477,0.024239,0.018344,0.004532
6,020_S_0213,ADNI_020_S_0213_MR_Hippocampal_Mask_Hi_2008022...,AD,63.0,0.144456,0.017821,0.013709,0.004856
7,020_S_0213,ADNI_020_S_0213_MR_Hippocampal_Mask_Hi_2008022...,AD,63.5,0.137800,0.026165,0.021022,0.003876
8,020_S_0213,ADNI_020_S_0213_MR_Hippocampal_Mask_Hi_2008022...,AD,64.0,0.138858,0.018875,0.015025,-0.006996
9,027_S_0120,ADNI_027_S_0120_MR_Hippocampal_Mask_Hi_2008040...,CN,72.0,0.137202,0.017233,0.013577,0.004020


In [7]:
for fig in train_helpers.train_velocity_figures(
    checkpoint=CHECKPOINT,
    device=DEVICE,
    n_subjects=N_SUBJECTS,
    seed=SEED,
    bin_width_years=BIN_WIDTH_YEARS,
    velocity_method=VELOCITY_METHOD,
):
    fig.show()


## Train adjacent-pair observed versus model change

This section compares the model's instantaneous speed field at the source scan against the observed adjacent-pair local change from the training data.


In [8]:
adjacent_payload = train_helpers.evaluate_selected_train_adjacent_pairs(
    checkpoint=CHECKPOINT,
    device=DEVICE,
    n_subjects=N_SUBJECTS,
    seed=SEED,
    bin_width_years=BIN_WIDTH_YEARS,
    velocity_method=VELOCITY_METHOD,
)

display(adjacent_payload['summary_df'].round(6))
display(
    adjacent_payload['compare_df'][[
        'subject_id',
        'diagnosis',
        'source_age_years',
        'target_age_years',
        'observed_area_weighted_rms_speed_per_year',
        'model_area_weighted_rms_speed_per_year',
        'local_speed_corr',
        'local_speed_rmse',
        'observed_net_volume_rate_per_year',
        'model_net_volume_rate_per_year',
    ]].round(6)
)


,cohort,num_rows,num_subjects,local_speed_corr_mean,local_speed_corr_ci_low,local_speed_corr_ci_high,local_speed_rmse_mean,local_speed_rmse_ci_low,local_speed_rmse_ci_high,observed_area_weighted_rms_speed_per_year_mean,...,observed_area_weighted_rms_speed_per_year_ci_high,model_area_weighted_rms_speed_per_year_mean,model_area_weighted_rms_speed_per_year_ci_low,model_area_weighted_rms_speed_per_year_ci_high,observed_net_volume_rate_per_year_mean,observed_net_volume_rate_per_year_ci_low,observed_net_volume_rate_per_year_ci_high,model_net_volume_rate_per_year_mean,model_net_volume_rate_per_year_ci_low,model_net_volume_rate_per_year_ci_high
0,all,40,20,0.196586,0.162611,0.228159,0.035029,0.032347,0.037922,0.032575,...,0.035622,0.019805,0.018701,0.020955,-0.004377,-0.008004,-0.001062,-0.000856,-0.003013,0.001622
1,CN,18,9,0.201205,0.146432,0.245302,0.037234,0.032868,0.042374,0.035351,...,0.040087,0.020230,0.018699,0.022019,-0.001596,-0.007467,0.004189,0.000629,-0.003148,0.004241
2,AD,22,11,0.192808,0.149429,0.238844,0.033224,0.030365,0.036497,0.030304,...,0.033349,0.019457,0.017985,0.020860,-0.006653,-0.009550,-0.003995,-0.002071,-0.004955,0.000737


,subject_id,diagnosis,source_age_years,target_age_years,observed_area_weighted_rms_speed_per_year,model_area_weighted_rms_speed_per_year,local_speed_corr,local_speed_rmse,observed_net_volume_rate_per_year,model_net_volume_rate_per_year
0,033_S_0733,AD,57.0,57.5,0.025571,0.016894,0.314038,0.026206,0.006876,0.003616
1,137_S_0366,AD,57.0,57.5,0.025427,0.015213,0.243529,0.027381,-0.001858,-0.002419
2,033_S_0733,AD,57.5,58.0,0.026988,0.020642,0.141554,0.032265,-0.001098,0.001932
3,137_S_0366,AD,57.5,58.0,0.028256,0.025766,0.295654,0.033638,-0.012932,0.006368
4,037_S_0627,AD,59.0,59.5,0.028778,0.013975,0.230946,0.028540,0.016781,0.003598
5,037_S_0627,AD,59.5,60.0,0.029827,0.019717,0.200404,0.032899,-0.024440,-0.001061
6,020_S_0213,AD,63.0,63.5,0.032359,0.017821,0.376071,0.031033,-0.012731,0.004856
7,020_S_0213,AD,63.5,64.0,0.023437,0.026165,-0.179102,0.038250,0.002659,0.003876
8,067_S_0029,AD,64.0,64.5,0.037580,0.017053,0.132656,0.038306,-0.015205,-0.000327
9,067_S_0029,AD,64.5,65.0,0.036103,0.020354,0.171846,0.037807,-0.015268,-0.014468


In [9]:
for fig in train_helpers.train_adjacent_comparison_figures(
    checkpoint=CHECKPOINT,
    device=DEVICE,
    n_subjects=N_SUBJECTS,
    seed=SEED,
    bin_width_years=BIN_WIDTH_YEARS,
    velocity_method=VELOCITY_METHOD,
):
    fig.show()


## Where the local change happens most on train subjects

These panels pick one adjacent train pair per diagnosis with relatively high local-speed agreement and show:

- observed local change
- model local change
- model minus observed


In [10]:
for fig in train_helpers.representative_train_local_map_figures(
    checkpoint=CHECKPOINT,
    device=DEVICE,
    n_subjects=N_SUBJECTS,
    seed=SEED,
    bin_width_years=BIN_WIDTH_YEARS,
    velocity_method=VELOCITY_METHOD,
):
    fig.show()


## In-distribution and out-of-distribution age sweeps from train latents

This chooses one representative `CN` and one representative `AD` start scan from the selected train subjects, starts from the learned training latent of that scan, and sweeps ages forward with both one-shot and yearly composed transport.

- in-distribution means age `<= 91`
- out-of-distribution means age `> 91`


In [11]:
sweep_payload = train_helpers.evaluate_train_age_sweeps(
    checkpoint=CHECKPOINT,
    device=DEVICE,
    n_subjects=N_SUBJECTS,
    seed=SEED,
    bin_width_years=BIN_WIDTH_YEARS,
    mesh_resolution=MESH_RESOLUTION,
    end_age_years=SWEEP_END_AGE,
    step_years=SWEEP_STEP_YEARS,
    reference_age_years=SWEEP_REFERENCE_AGE,
)

display(
    sweep_payload['representative_start_rows'][[
        'subject_id',
        'scan_id',
        'diagnosis',
        'continuous_age_years',
        'visit_order',
    ]].round(3)
)
display(sweep_payload['summary_df'].round(6))
display(
    sweep_payload['sweep_df'][[
        'subject_id',
        'diagnosis',
        'method',
        'age_years',
        'in_distribution',
        'mesh_valid',
        'pred_volume',
        'probe_sdf_min',
        'probe_sdf_max',
        'latent_l2_norm',
    ]].head(40).round(6)
)


ERROR:root:[create_mesh] Caught marching cubes error: Surface level must be within volume data range..
ERROR:root:[create_mesh] Caught marching cubes error: Surface level must be within volume data range..
ERROR:root:[create_mesh] Caught marching cubes error: Surface level must be within volume data range..
ERROR:root:[create_mesh] Caught marching cubes error: Surface level must be within volume data range..
ERROR:root:[create_mesh] Caught marching cubes error: Surface level must be within volume data range..
ERROR:root:[create_mesh] Caught marching cubes error: Surface level must be within volume data range..
ERROR:root:[create_mesh] Caught marching cubes error: Surface level must be within volume data range..
ERROR:root:[create_mesh] Caught marching cubes error: Surface level must be within volume data range..
ERROR:root:[create_mesh] Caught marching cubes error: Surface level must be within volume data range..
ERROR:root:[create_mesh] Caught marching cubes error: Surface level must 

,subject_id,scan_id,diagnosis,continuous_age_years,visit_order
0,057_S_0643,ADNI_057_S_0643_MR_Hippocampal_Mask_Hi_2008040...,CN,71.0,0
1,036_S_1001,ADNI_036_S_1001_MR_Hippocampal_Mask_Hi_2008022...,AD,70.0,2


,subject_id,diagnosis,start_scan_id,start_age_years,method,last_valid_age_years,first_invalid_age_years
0,036_S_1001,AD,ADNI_036_S_1001_MR_Hippocampal_Mask_Hi_2008022...,70.0,composed,79.0,80.0
1,036_S_1001,AD,ADNI_036_S_1001_MR_Hippocampal_Mask_Hi_2008022...,70.0,direct,79.0,80.0
2,057_S_0643,CN,ADNI_057_S_0643_MR_Hippocampal_Mask_Hi_2008040...,71.0,composed,78.0,79.0
3,057_S_0643,CN,ADNI_057_S_0643_MR_Hippocampal_Mask_Hi_2008040...,71.0,direct,78.0,79.0


,subject_id,diagnosis,method,age_years,in_distribution,mesh_valid,pred_volume,probe_sdf_min,probe_sdf_max,latent_l2_norm
0,036_S_1001,AD,composed,70.0,True,True,0.096219,-0.112973,0.876325,0.548551
1,036_S_1001,AD,composed,71.0,True,True,0.093964,-0.096475,0.879239,0.571708
2,036_S_1001,AD,composed,72.0,True,True,0.096950,-0.064416,0.874984,0.615988
3,036_S_1001,AD,composed,73.0,True,True,0.068211,-0.048174,0.866254,0.681609
4,036_S_1001,AD,composed,74.0,True,True,0.062879,-0.062960,0.854289,0.764651
5,036_S_1001,AD,composed,75.0,True,True,0.060728,-0.065763,0.846121,0.862790
6,036_S_1001,AD,composed,76.0,True,True,0.038012,-0.028393,0.857256,0.973939
7,036_S_1001,AD,composed,77.0,True,True,0.016504,-0.023915,0.871945,1.096725
8,036_S_1001,AD,composed,78.0,True,True,0.003676,-0.003441,0.877502,1.230018
9,036_S_1001,AD,composed,79.0,True,True,0.000226,0.002323,0.884738,1.373288


In [12]:
for fig in train_helpers.train_age_sweep_figures(
    checkpoint=CHECKPOINT,
    device=DEVICE,
    n_subjects=N_SUBJECTS,
    seed=SEED,
    bin_width_years=BIN_WIDTH_YEARS,
    mesh_resolution=MESH_RESOLUTION,
    end_age_years=SWEEP_END_AGE,
    step_years=SWEEP_STEP_YEARS,
    reference_age_years=SWEEP_REFERENCE_AGE,
):
    fig.show()
